In [ ]:
pip install -U transformers accelerate bitsandbytes peft trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 55.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


# Import Libraries

In [ ]:
import os, json, torch
import pandas as pd
from datasets import Dataset
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel, TaskType

import os
os.environ["WANDB_DISABLED"] = "true"


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Configuration

In [ ]:
Model = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Train_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/train_balanced.csv"
Val_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/val.csv"
Test_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/test.csv"

Labels = ["Normal", "Bipolar/Personality", "Anxiety/Stress", "Depressive_Spectrum"]

# Data Preparation

In [ ]:
def build_example(text, label=None):
    system = "You are a helpful classifier. Reply with exactly one label from: " + ", ".join(Labels) + "."
    user = f"Text: {text}\nLabel options: " + ", ".join(Labels) + "\nAnswer with one label only."
    return {
        "system": system,
        "user": user,
        "assistant": label or "",
    }


def df_to_hf(df, is_train=True): # converts df into a Hugging Face dataset
    recs = []
    for _, row in df.iterrows():
        recs.append(build_example(row["statement"], row["status_combined"] if is_train else None))
    return Dataset.from_list(recs)


train_df = pd.read_csv(Train_path)
val_df = pd.read_csv(Val_path)
train_ds = df_to_hf(train_df)
val_ds = df_to_hf(val_df)

In [ ]:
print(train_ds[:2]["system"])

['You are a helpful classifier. Reply with exactly one label from: Normal, Bipolar/Personality, Anxiety/Stress, Depressive_Spectrum.', 'You are a helpful classifier. Reply with exactly one label from: Normal, Bipolar/Personality, Anxiety/Stress, Depressive_Spectrum.']


In [ ]:
print(f"[Step 2] Train rows={len(train_df):,}, Val rows={len(val_df):,}")
print(f"[Step 2] Label examples: {train_df['status_combined'].value_counts().to_dict()}")

[Step 2] Train rows=44,912, Val rows=9,624
[Step 2] Label examples: {'Anxiety/Stress': 11228, 'Normal': 11228, 'Bipolar/Personality': 11228, 'Depressive_Spectrum': 11228}


# Tokenizer + Quantization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

# LoRA Configuration

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
)

train_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_steps=1000,
    bf16=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    packing=False,
    dataset_num_proc=2,
    report_to="none",
    logging_dir="./logs",
)


# Training

In [ ]:
def formatting_func(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    # Convert chat messages to a single text sequence (no tokenization yet)
    return tokenizer.apply_chat_template(messages, tokenize=False)

trainer = SFTTrainer(
    model=Model,                 # or model name string
    peft_config=peft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=train_cfg,
    formatting_func=formatting_func,  # ✅ single-example formatter
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Applying formatting function to train dataset (num_proc=2):   0%|          | 0/44912 [00:00<?, ? examples/s]

Adding EOS to train dataset (num_proc=2):   0%|          | 0/44912 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/44912 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/44912 [00:00<?, ? examples/s]

Applying formatting function to eval dataset (num_proc=2):   0%|          | 0/9624 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=2):   0%|          | 0/9624 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/9624 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=2):   0%|          | 0/9624 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

No checkpoint found; starting fresh.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.893100,1.765159,1.804564,1593634.000000,0.619771
1000,1.773400,1.675359,1.698117,3217018.000000,0.637226
1500,1.642900,1.592571,1.566949,4824441.000000,0.654122
2000,1.513200,1.504717,1.590845,6433314.000000,0.672569
2500,1.435100,1.420689,1.462511,8017591.000000,0.690478
3000,1.146000,1.355015,1.383402,9611319.000000,0.707150
3500,1.109200,1.298668,1.292171,11214425.000000,0.722532
4000,1.083600,1.244025,1.234812,12839501.000000,0.736245
4500,1.031200,1.197133,1.203021,14455777.000000,0.747455
5000,0.953400,1.171937,1.192964,16046424.000000,0.753798


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.893100,1.765159,1.804564,1593634.000000,0.619771
1000,1.773400,1.675359,1.698117,3217018.000000,0.637226
1500,1.642900,1.592571,1.566949,4824441.000000,0.654122
2000,1.513200,1.504717,1.590845,6433314.000000,0.672569
2500,1.435100,1.420689,1.462511,8017591.000000,0.690478
3000,1.146000,1.355015,1.383402,9611319.000000,0.707150
3500,1.109200,1.298668,1.292171,11214425.000000,0.722532
4000,1.083600,1.244025,1.234812,12839501.000000,0.736245
4500,1.031200,1.197133,1.203021,14455777.000000,0.747455
5000,0.953400,1.171937,1.192964,16046424.000000,0.753798


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


('/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/tokenizer.model',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/mistral_cls_adapter/tokenizer.json')